<a href="https://colab.research.google.com/github/benjaminsheppard/Event_Review_Toolkit/blob/main/CO_timelapse_radar_ffmpeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Code Block 1 -- Initial Setup {"display-mode":"form"}
# @markdown Welcome to the Radar Timelapse .MP4 Video Maker by Ben Sheppard,
# @markdown formerly known as "AnimateMP4" on your Windows desktop.
# @markdown
# @markdown This script will walk you through the process of creating a .mp4
# @markdown video animation from a folder of .png radar images.
# @markdown
# @markdown To begin, hover over this text and press the play button (triangle)
# @markdown on the top left of this code block. This will setup everything the
# @markdown code needs in order to run. When the green check-mark appears,
# @markdown scroll down and read the instructions.

!apt-get install -y ffmpeg
!mkdir png_frames

import sys, os, glob
from datetime import datetime, timedelta

print("\n\n--> Setup complete! Scroll down to the next code block.")

**Instructions:**

Click on the Folder icon in the toolbar at left, this will display the "Files" view. Assuming Code Block 1 has successfully executed, there should be a folder titled "png_frames" visible. Hover over it to reveal the triple-dot menu, click it and select "Upload". This is where you will "Upload" your .png image frames from your computer. Ideally, these .png files (radar frames) should be saved using the naming scheme demonstrated below...<br>
kmkx_20251109_2305_BR_0.5.png<br>
which basically boils down to...<br>
`<Radar Site ID>_<YYYYMMDD Date>_<HHMM Time>_<Base Product Displayed>_<Tilt Elev>.png`<br>
<br>
**Note:** For operating mode 2, the naming scheme above is required. For operating mode 1, the only file name requirements are as follows...

   - Must be a valid unix filename.
   - Must sort properly (consistent alphanumeric order).
   - All filenames must be the same character length.
   - Valid Example: 001.png, 002.png, 003.png, etc..., 150.png

<br>

---

<br>

Now you are ready to run Code Block 2 (below) via hovering over it and clicking the play button (triangle). As Code Block 2 runs, it will ask you for inputs (beneath the code block).

Select an operating mode by typing in a single integer (whole number) and pressing enter. Your options are...
   1. Make a video with a fixed frame rate.
   2. Make a video in timelapse mode, where the frame rate varies to account for faster / slower radar scan rates (VCP and AVSET changes).
   3. Make a video in timelapse mode, but use the hardcoded frame delays from radar_framedelays.txt (assuming that you've run mode 2 already and made manual edits to radar_framedelays.txt).

Depending on what mode you select, you may be prompted to enter additional inputs to the code...
   - Runtime (sec): How many seconds long do you want the video to be? Enter a whole number (integer), no decimals!
   - Last-Frame Dwell (sec): How many seconds do you want to pause on the last frame? Zero is not an option, enter a whole number (integer), no decimals!

When the code has all the inputs it needs, it will run automatically and produce a file called animation.mp4, located in the same folder as the .png images (png_frames). Expand the folder to view it's contents, hover over "animation.mp4", then from the triple-dot menu select "Download".

If you wish to speed up, slow down, or otherwise change the animation, note that you can simply re-run Code Block 2 with new settings. You can then re-download animation.mp4.

**Warning:** <br>
When you close out of this browser tab, or leave it sitting inactive for more than 5-15 minutes, the "Runtime will be disconnected", which is just a fancy way of saying all the files in the "Files" view will be permanently deleted and all of the code outputs / printouts will be cleared away. This is a Google Colab feature (not a bug), which reduces unnecessary burden on Google's servers **and** ensures that the next time you use this tool it's fresh and ready to go (rather than having your old .png files still in there). Bottom line is, when you're done making an animation, be sure to download it and keep it (rather than leaving it here where it will be destroyed).

In [ ]:
# @title Code Block 2 -- Animation Generation {"display-mode":"form"}
# @markdown <-- Use this triangle button to start. Additional user prompts and
# @markdown instructions will print out below.
def datetime_from_filepath(filePath):
    fileName = os.path.basename(filePath)
    fileParts = fileName.split("_")
    ymd = fileParts[1]
    hm  = fileParts[2]
    timestamp = datetime.strptime(ymd+hm, "%Y%m%d%H%M")
    return timestamp


def main_program():
    # Gather interactive user inputs...
    mode = int(input("Enter Operating Mode: "))
    if mode not in [1, 2, 3]:
        print("Fatal Error: Unsupported operating mode.")
        raise Exception("Program Stop.")
    homeDir = "/content/png_frames"
    if mode in [1, 2]:
        reqRuntime_sec = int(input("Runtime (sec): "))
    if mode == 2:
        lastFrameDwell_sec = int(input("Last-Frame Dwell (sec): "))
    print()
    # Locate, list, and sort the .png files for the animation...
    fileList = glob.glob(os.path.join(homeDir, "*.png"))
    fileList.sort() # Sort in-place alphanumerically ascending.
    if len(fileList) < 2:
        print("Fatal Error: Not enough .png frames found.")
        raise Exception("Program Stop.")
    # Predefine the location of the config file and output video...
    outFullFile = os.path.join(homeDir, "animation.mp4")
    cfgFullFile = os.path.join(homeDir, "radar_framedelays.txt")
    # Ensure the output animation file does not exist yet...
    if os.path.isfile(outFullFile):
        os.remove(outFullFile)
    # If we're using mode 1 or 2, ensure the config file does not exist yet...
    if mode in [1, 2]:
        if os.path.isfile(cfgFullFile):
            os.remove(cfgFullFile)
    # For mode 3, leave the config file intact and use it.

    # We've got all the inputs we need! Execute the operating mode...
    if mode == 1:
        print("Running with Operating Mode 1...")
        # Compute the ideal frame delay...
        animDelay = str(round((reqRuntime_sec / len(fileList)) * 100000) / 100000)
        # Write out a list of filenames and time delays (for FFMPEG)
        cfgFile = open(cfgFullFile, "w")
        cfgFile.write("ffconcat version 1.0\n")
        for file1 in fileList:
            # Add this info to radar_framedelays.txt
            cfgFile.write(f"file '{file1}'\n")
            cfgFile.write(f"duration {animDelay}\n")
        # Write the last frame AGAIN without any duration.
        # FFMPEG requires this for some reason.
        cfgFile.write(f"file '{file1}'\n")
        # Close out the config file.
        cfgFile.close()
        print("radar_framedelays.txt loaded and saved.")
        # Create the animation generation code (FFMPEG) for this mode.
        sysCmd = f'''ffmpeg -f concat -safe 0 -i "{cfgFullFile}" -c:v libx264 -pix_fmt yuv420p -vf "scale='iw-mod(iw,2)':'ih-mod(ih,2)'" "{outFullFile}"'''

    elif mode == 2:
        print("Running with Operating Mode 2...")
        # Measure the length of "real-life time elapsed" in the radar frames...
        tStart = datetime_from_filepath(fileList[ 0])
        tEnd   = datetime_from_filepath(fileList[-1])
        tElapsed = tEnd - tStart
        tElapsed = tElapsed.total_seconds()
        # Write out a list of filenames and time delays (for FFMPEG)
        cfgFile = open(cfgFullFile, "w")
        cfgFile.write("ffconcat version 1.0\n")
        for j in range(1, len(fileList)):
            i = j - 1
            file1 = fileList[i]
            file2 = fileList[j]
            time1 = datetime_from_filepath(file1)
            time2 = datetime_from_filepath(file2)
            timeDelta = time2 - time1
            timeDelta = timeDelta.total_seconds()
            # How many seconds should we spend on this frame?
            # Round to 5 places after the decimal.
            animDelay = str(round((timeDelta / tElapsed) * reqRuntime_sec * 100000) / 100000)
            # Add this info to radar_framedelays.txt
            cfgFile.write(f"file '{file1}'\n")
            cfgFile.write(f"duration {animDelay}\n")
        # Write in the last frame with it's own dwell time.
        cfgFile.write(f"file '{file2}'\n")
        cfgFile.write(f"duration {lastFrameDwell_sec}\n")
        # Write the last frame AGAIN without any duration.
        # FFMPEG requires this for some reason.
        cfgFile.write(f"file '{file2}'\n")
        # Close out the config file.
        cfgFile.close()
        print("radar_framedelays.txt loaded and saved.")
        # Create the animation generation code (FFMPEG) for this mode.
        sysCmd = f'''ffmpeg -f concat -safe 0 -i "{cfgFullFile}" -c:v libx264 -pix_fmt yuv420p -vsync vfr -vf "scale='iw-mod(iw,2)':'ih-mod(ih,2)'" "{outFullFile}"'''

    elif mode == 3:
        print("Running with Operating Mode 3...")
        # All we've got to do is rerun FFMPEG with whatever's in the preexisting config file.
        sysCmd = f'''ffmpeg -f concat -safe 0 -i "{cfgFullFile}" -c:v libx264 -pix_fmt yuv420p -vsync vfr -vf "scale='iw-mod(iw,2)':'ih-mod(ih,2)'" "{outFullFile}"'''

    # Regardless of what mode we're in, the end goal is to run FFMPEG and produce animation.png...
    print("Running FFMPEG...")
    print(sysCmd)
    !{sysCmd}
    # Done!
    print("\nFFMPEG command completed, check for 'animation.mp4' in the same folder as the .png files.\n")


if __name__ == '__main__':
    main_program()


